# DS2002 · Capstone Kickoff and First Pulls

**Studio — 2026-11-18 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone kickoff and first pulls

The capstone starts today. Read `DS2002 Capstone — ACC Reality Check.pdf` in this folder before Friday, because it describes everything your team will hand in and when. The short version is that your team will decide which ACC men's basketball teams are as good as their records say and which ones have built their records on easy opponents. To do that you will combine a ratings file with game results from a live API, keep everything in a SQLite database, answer the questions with SQL, and publish your results as a small web API.

Today has a modest goal. Each team forms, sets up a repository, and looks at both sources with its own eyes. You will not clean or load anything yet. The point of the hour is to find out what the data looks like before anyone writes a plan around it.

If you do not follow basketball, start with the appendix of the PDF. It explains every term used in this notebook, and nobody on your team should feel behind because of the sport.

### Step 1: Form your team

Teams have three or four people. Fill in the cell below together and commit it to the team repository by the end of class. The roles are a starting point, and you can trade them later, but someone should own each part of the project from the first day.

In [1]:
team = {
    'name': 'The Cleaners',
    'members': ['Karen Tran', 'Zach Bernas'],
    'repo_url': 'https://github.com/karen-tran08/ds2002-acc-reality-check',
    'roles': {
        'ratings CSV and crosswalk': 'Karen',
        'ESPN pulls and box scores': 'Zach',
        'SQLite schema and queries': 'Karen',
        'API and PythonAnywhere': 'Zach',
    },
    'meeting_time': 'Monday, 2PM',
}

unfilled = [k for k, v in team.items() if 'TODO' in str(v)]
print('still unfilled:', unfilled if unfilled else 'nothing, the charter is complete')

still unfilled: nothing, the charter is complete


### Step 2: Pull the ratings CSV

Bart Torvik publishes a rating for every Division I team as a CSV file at this address:

`https://barttorvik.com/2027_team_results.csv`

The 2027 in the name refers to the season that ends in 2027, which is the season being played right now. You do not need a key or an account.

**TODO:** download the file with `requests`, read it into pandas, and look at it before you do anything else. Print the shape, the column names, and the first few rows. Save the raw text to a file named with today's date, because the ratings change every day and you will want to know which version you used.

In [2]:
import requests
import pandas as pd
from io import StringIO
from datetime import date

url = "https://barttorvik.com/2027_team_results.csv"
fname = f"torvik_2027_team_results_{date.today().isoformat()}.csv"

try:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    raw_text = resp.text
    source = "requests"
except requests.HTTPError as e:
    print(f"Request blocked ({e}). Upload the CSV you downloaded in your browser:")
    from google.colab import files
    uploaded = files.upload()   # pick 2027_team_results.csv
    raw_text = next(iter(uploaded.values())).decode("utf-8")
    source = "manual browser download"

# Save the raw text with today's date
with open(fname, "w", encoding="utf-8") as f:
    f.write(raw_text)
print(f"Saved {fname} (source: {source})")

df = pd.read_csv(StringIO(raw_text))
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Request blocked (403 Client Error: Forbidden for url: https://barttorvik.com/2027_team_results.csv). Upload the CSV you downloaded in your browser:


Saving 2027_team_results.csv to 2027_team_results (1).csv
Saved torvik_2027_team_results_2026-11-20.csv (source: manual browser download)
Shape: (365, 45)
Columns: ['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.068109,3,92.914704,3,0.965284,1,...,0.0,0.0,0,0,0.843419,0,2.494921,1,27,67.979435
1,2,Kansas,B12,5-1,121.915951,6,93.629712,4,0.954166,2,...,0.0,0.0,0,0,0.840712,0,1.376823,5,160,73.248990
2,3,Duke,ACC,3-2,119.914227,14,92.135429,1,0.953931,3,...,0.0,0.0,0,0,0.808412,0,-0.448474,118,353,71.755678
3,4,Florida,SEC,6-1,124.257078,2,95.554648,12,0.953493,4,...,0.0,0.0,0,0,0.857483,0,2.345710,2,75,69.045058
4,5,Illinois,B10,4-1,124.838970,1,96.482622,19,0.950879,5,...,0.0,0.0,0,0,0.838993,0,0.976552,15,110,74.199610


**TODO:** answer these in the cell below, using what you printed.

1. How many rows are there, and how many belong to the ACC? You should be able to filter on one column to find out.
2. Two of the original headers have the same name. What did pandas call them, and which one would you trust as the team's rank?
3. Look at how a few schools are spelled, such as North Carolina State and Miami. Write the spellings down exactly, because the other source will spell them differently.

In [11]:
# Q1
print("Total rows:", len(df))
acc = df[df["conf"] == "ACC"]
print("ACC rows:", len(acc))

# Q2
dupes = [c for c in df.columns if "." in c and c.rsplit(".", 1)[-1].isdigit()]
print("Renamed duplicate columns:", dupes)
for d in dupes:
    orig = d.rsplit(".", 1)[0]
    print(df[[orig, d, "team"]].head(10))

# Q3
for name in acc["team"].sort_values():
    print(repr(name))

Total rows: 365
ACC rows: 18
Renamed duplicate columns: ['rank.1']
   rank  rank.1          team
0     1       1       Houston
1     2       2        Kansas
2     3       3          Duke
3     4       4       Florida
4     5       5      Illinois
5     6       6       Arizona
6     7       7      Virginia
7     8       8  Michigan St.
8     9       9   Connecticut
9    10      10    Texas Tech
'Boston College'
'California'
'Clemson'
'Duke'
'Florida St.'
'Georgia Tech'
'Louisville'
'Miami FL'
'N.C. State'
'North Carolina'
'Notre Dame'
'Pittsburgh'
'SMU'
'Stanford'
'Syracuse'
'Virginia'
'Virginia Tech'
'Wake Forest'


In [4]:
print(df[["rank", "rank.1"]].dtypes)
diff = df[df["rank"] != df["rank.1"]]
print("Rows where they differ:", len(diff))
print(diff[["rank", "rank.1", "team"]].head(10))
print("rank is 1..365 in order:", (df["rank"] == range(1, len(df) + 1)).all())

rank      int64
rank.1    int64
dtype: object
Rows where they differ: 0
Empty DataFrame
Columns: [rank, rank.1, team]
Index: []
rank is 1..365 in order: True


1. There are 365 total rows and 18 belong to the ACC.
2. The duplicate header was called "rank.1." Both are integers and they match across all 365 rows, so the duplicate header is just a copy. So, I'm going to drop the duplicate header and trust the first.
3. These are the following spellings for the schools:

'Boston College'
'California'
'Clemson'
'Duke'
'Florida St.'
'Georgia Tech'
'Louisville'
'Miami FL'
'N.C. State'
'North Carolina'
'Notre Dame'
'Pittsburgh'
'SMU'
'Stanford'
'Syracuse'
'Virginia'
'Virginia Tech'
'Wake Forest'

### Step 3: Pull one day of games from ESPN

ESPN serves game results as JSON with no key. This address returns every game an ACC team played on one date:

`https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard`

Pass the parameters `groups=2` (the ACC), `dates=YYYYMMDD`, and `limit=200`. Pick a date from the past week. You practiced this kind of request in October, so reuse what worked for you then, including a timeout and a check of the status code.

**TODO:** make the request, then explore the JSON one level at a time. Find the list of games, count them, and for one game print both team names, both scores, whether the game is finished, and the start time exactly as ESPN wrote it.

In [5]:
# TODO: request one date, then find the games inside the JSON
import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
date_pulled = "20261117"   # pick any date from the past week
params = {"groups": 2, "dates": date_pulled, "limit": 200}

resp = requests.get(url, params=params, timeout=30)
print("Status:", resp.status_code)
resp.raise_for_status()

data = resp.json()
print(type(data))
print("Top-level keys:", list(data.keys()))

Status: 200
<class 'dict'>
Top-level keys: ['leagues', 'groups', 'events', 'provider', 'eventsDate']


In [6]:
games = data["events"]
print("Games that day:", len(games))

game = games[0]
print("Keys in one game:", list(game.keys()))
print("Name:", game["name"])

Games that day: 10
Keys in one game: ['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']
Name: Notre Dame Fighting Irish at Florida Gators


In [7]:
comp = game["competitions"][0]
print("Keys in competition:", list(comp.keys()))

for team in comp["competitors"]:
    print(team["homeAway"], "|",
          team["team"]["displayName"], "|",
          team["team"]["shortDisplayName"], "|",
          "score:", team["score"])

status = game["status"]["type"]
print("Finished?", status["completed"], "|", status["description"])
print("Start time (as ESPN wrote it):", repr(game["date"]))

Keys in competition: ['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'startDate', 'broadcast', 'geoBroadcasts', 'highlights', 'headlines']
home | Florida Gators | Florida | score: 86
away | Notre Dame Fighting Irish | Notre Dame | score: 78
Finished? True | Final
Start time (as ESPN wrote it): '2026-11-17T21:51Z'


In [8]:
for g in games:
    c = g["competitions"][0]["competitors"]
    teams = " vs ".join(f'{t["team"]["shortDisplayName"]} {t["score"]}' for t in c)
    print(g["date"], "|", teams, "|", g["status"]["type"]["description"])

2026-11-17T21:51Z | Florida 86 vs Notre Dame 78 | Final
2026-11-18T02:00Z | Virginia 85 vs Old Dominion 65 | Final
2026-11-18T00:00Z | Duke 78 vs Illinois 99 | Final
2026-11-18T00:00Z | Miami 95 vs Rider 48 | Final
2026-11-18T00:00Z | Georgia Tech 93 vs Norfolk St 79 | Final
2026-11-18T00:00Z | Wake Forest 81 vs Monmouth 76 | Final
2026-11-18T00:00Z | Pitt 81 vs Villanova 61 | Final
2026-11-18T00:00Z | Florida St 85 vs Ball State 71 | Final
2026-11-18T01:00Z | SMU 76 vs Oral Roberts 72 | Final
2026-11-18T03:00Z | Stanford 65 vs UC Irvine 61 | Final


In [12]:
torvik_names = set(df["team"])
opponents = ["Old Dominion", "Illinois", "Rider", "Norfolk St", "Monmouth",
             "Villanova", "Ball State", "Oral Roberts", "UC Irvine", "Florida"]
print(df[df["team"].str.contains("Norfolk|Ball", case=False)][["team", "conf"]])

            team  conf
295     Ball St.   MAC
348  Norfolk St.  MEAC


**TODO:** answer these in the cell below.

1. Where in the JSON are the games, and where inside one game are the two teams?
2. Compare the start time ESPN gives for an evening game with the date you asked for. What time zone is the start time in, and what goes wrong if you use its first ten characters as the game date?
3. How does ESPN spell the schools you wrote down in Step 2?
4. Did every opponent on your date play in Division I? Look the unfamiliar names up.

1. Inside one game, the two teams are in game["competitions"][0]["competitors"]. Each competitor has homeAway, team (with the names) and score. The games are in data["events"], a list of 10 games.
2. 9 of 10 games are stamped 2026-11-18. For example, 2026-11-18T00:00Z is 7:00 PM Eastern on Nov 17. Only the Florida–Notre Dame game (21:51Z, about 4:51 PM ET) shows Nov 17. If you take the first 10 characters as the game date, almost every evening game lands on the wrong day, so we need to convert to Eastern time first. The start time is in UTC (Z at the end).
3. ESPN spellings vs. Torvik:
    - Miami vs. Torvik Miami FL
    - Pitt vs. Torvik Pittsburgh
    - Florida St vs. Torvik Florida St.
    - SMU, Duke, Virginia, and others match
    - N.C. State didn't play on Nov 17, so its ESPN spelling isn't in this pull
    - displayName adds the mascot, (e.g. "Notre Dame Fighting Irish"), so it's even further from Torvik's spelling
4. All 10 opponents are Division I. Checking ESPN’s names against Torvik’s 365 teams, 8 matched exactly. Norfolk St and Ball State didn’t match because Torvik spells them Norfolk St. and Ball St. They’re the same schools, so they need crosswalk entries.

### Step 4: Write down the work ahead

You have now seen both sources. Before you leave, list at least five specific things that will have to be cleaned, converted, or matched before the two sources can be joined, and make each one concrete. "Names are different" is less useful than "ESPN writes NC State and Torvik writes N.C. State." This list becomes the first entries in your decision log on Friday.

_1._ School names differ between sources. ESPN’s shortDisplayName writes Pitt, Miami and Florida St, while Torvik writes Pittsburgh, Miami FL and Florida St. (with a period). We need a crosswalk table that maps each ESPN name to its Torvik name before joining.

_2._ ESPN start times are in UTC, not Eastern. On 2026-11-17, 9 of 10 games are stamped 2026-11-18 (e.g. 2026-11-18T00:00Z = 7:00 PM ET on Nov 17). Convert date to America/New_York before taking the game date, or evening games land on the wrong day.

_3._ Torvik has a duplicate rank column. pandas renamed the second one rank.1. It’s identical to rank on all 365 rows, so drop rank.1 and keep rank.

_4._ Torvik’s record is text, not numbers. Values look like "6-0". Split it into integer wins and losses columns so we can compare records to ratings in SQL.

_5._ Torvik blocks scripted downloads from Colab (403). We downloaded the CSV in a browser and saved it as torvik_2027_team_results_2026-11-20.csv. Ratings change daily, so every pull needs a dated file name, and the pipeline needs a plan for getting the file (manual download or running locally).


### Before Friday

Friday's check-in notebook walks your team through getting the first version of the pipeline running, including the name crosswalk and the first tables in SQLite. Read the PDF tonight, and agree as a team on who is starting which part before you leave the room.

### Team progress

This cell is for your team, and nothing is submitted or graded. The only graded work in the capstone is the final submission on Friday, December 18, so these progress cells exist to help you notice early when something is behind. Fill it in and commit it with the rest of the notebook.

In [10]:
progress = {
    'acc_rows_in_torvik': 18,     # TODO: how many ACC rows the CSV has
    'espn_date_pulled': '20261117',     # TODO: the date you asked ESPN for
    'espn_games_that_day': 10,    # TODO: how many games came back
    'name_that_differs': 'Pitt (ESPN) vs Pittsburgh (Torvik)',    # TODO: one school spelled two ways
    'blocker': 'Torvik returns 403 to Colab; downloaded CSV manually',              # TODO: what is in your way, or 'nothing yet'
}

for k, v in progress.items():
    print(f'{k}: {v}')

acc_rows_in_torvik: 18
espn_date_pulled: 20261117
espn_games_that_day: 10
name_that_differs: Pitt (ESPN) vs Pittsburgh (Torvik)
blocker: Torvik returns 403 to Colab; downloaded CSV manually
